# Project 4: AI Product Requirement Generator

This Jupyter Notebook demonstrates the core functionality of the **AI Product Requirement Generator** using a multi-agent system. The system runs in a pipeline format:
1. **Input**: Product Idea (e.g., "Build a mobile healthcare platform").
2. **Business Analyst Agent**: Generates the Problem Statement, Target Personas, and Risks & Mitigations.
3. **Product Manager Agent**: Takes the BA's analysis and translates it into User Stories, Functional Requirements, and Non-functional Requirements.
4. **Database (MongoDB)**: Stores the generated outputs and details for persistence and history tracking.

Let's walk through the implementation step by step.

### Step 1: Environment Setup and Imports

First, we make sure we load our environment variables (e.g., `GEMINI_API_KEY`, `MONGO_URI`) and import our core classes from `prd_generator.py`.

In [15]:
import os
import json
from dotenv import load_dotenv
from prd_generator import LLMService, BusinessAnalystAgent, ProductManagerAgent, DatabaseManager

# Load environment variables from .env file if present
load_dotenv()

print("Libraries imported successfully.")
print(f"GEMINI_API_KEY configured: {bool(os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'))}")
print(f"MONGO_URI configured: {os.environ.get('MONGO_URI', 'Not configured (using default: localhost)')}")

Libraries imported successfully.
GEMINI_API_KEY configured: False
MONGO_URI configured: Not configured (using default: localhost)


### Step 2: Initialize Agents and Services

We instantiate our LLM Service and the two specialized agents (Business Analyst and Product Manager), as well as the Database Manager for MongoDB integration.

In [16]:
# Initialize LLM service (will automatically use google-genai or google-generativeai)
llm_service = LLMService()

# Initialize agents
ba_agent = BusinessAnalystAgent(llm_service)
pm_agent = ProductManagerAgent(llm_service)

# Initialize MongoDB Database Manager
db_manager = DatabaseManager()
print(f"MongoDB Connection Successful: {db_manager.is_connected}")

2026-06-25 17:49:26,525 - WARNING - No Gemini API key found in GEMINI_API_KEY or GOOGLE_API_KEY env variables.
2026-06-25 17:49:26,531 - INFO - Successfully connected to MongoDB.


MongoDB Connection Successful: True


### Step 3: Run the Business Analyst Agent

We provide the product idea, target audience, and constraints, then invoke the Business Analyst Agent to generate the initial structured analysis.

In [17]:
# Define product inputs
idea = "Build a mobile healthcare platform"
target_audience = "Patients seeking remote doctor consultations and prescription tracking"
constraints = "Must be HIPAA compliant, run on iOS and Android"

print(f"--- Running Business Analyst Agent for: '{idea}' ---")
ba_result = ba_agent.analyze(idea=idea, target_audience=target_audience, constraints=constraints)

# Print raw JSON response
print(json.dumps(ba_result, indent=2))

2026-06-25 17:49:26,539 - INFO - Providing mock fallback data...


--- Running Business Analyst Agent for: 'Build a mobile healthcare platform' ---
{
  "user_stories": [
    {
      "title": "Enter Product Idea",
      "role": "Product Manager",
      "want": "to input a high-level product idea with specific target constraints",
      "benefit": "I can quickly initiate a comprehensive requirements generation workflow",
      "acceptance_criteria": "1. Input field must support up to 1000 characters.\n2. Optional inputs for target audience and constraints should be available.\n3. Validate that input is not empty before submitting."
    },
    {
      "title": "View Sectioned Output",
      "role": "Product Designer",
      "want": "to view generated requirements split into tabbed sections (personas, requirements, risks)",
      "benefit": "I can review specific aspects of the PRD easily without reading a wall of text",
      "acceptance_criteria": "1. UI should display distinct tabs for Personas, Stories, Requirements, and Risks.\n2. Elements should use

### Step 4: Run the Product Manager Agent

Now we pass the Business Analyst's output into the Product Manager Agent. The PM agent translates the personas and problem statement into user stories, functional requirements, and non-functional requirements.

In [18]:
print(f"--- Running Product Manager Agent ---")
pm_result = pm_agent.analyze(idea=idea, ba_analysis=ba_result)

# Print raw JSON response
print(json.dumps(pm_result, indent=2))

2026-06-25 17:49:26,545 - INFO - Providing mock fallback data...


--- Running Product Manager Agent ---
{
  "problem_statement": "The existing workflow for product idea realization is highly fragmented. Teams face issues in translating high-level vision into functional requirements, resulting in misalignment, missed details, and prolonged launch delays.",
  "personas": [
    {
      "name": "Sarah Miller",
      "role": "Product Leader",
      "details": "Manage multiple roadmap priorities in a fast-paced environment. Wants structured specifications quickly to align developers.",
      "pain_points": "Spends hours writing PRDs from scratch; developers complain about missing edge cases.",
      "goals": "Generate high-quality initial requirements in minutes to jumpstart development sprint planning."
    },
    {
      "name": "Alex Chen",
      "role": "Tech Lead / Architect",
      "details": "Technical owner responsible for sizing features, planning sprints, and ensuring system design aligns with product requirements.",
      "pain_points": "Vague u

### Step 5: Save PRD to MongoDB Database

Let's save our compiled PRD (inputs, BA data, PM data) to MongoDB using our `DatabaseManager`.

In [19]:
print("--- Saving PRD to Database ---")
prd_id = db_manager.save_prd(
    idea=idea,
    target_audience=target_audience,
    constraints=constraints,
    ba_data=ba_result,
    pm_data=pm_result
)
print(f"PRD saved successfully with Database ID: {prd_id}")

--- Saving PRD to Database ---
PRD saved successfully with Database ID: 6a3d1cce2fcde7616bc1c1ac


### Step 6: Verify Persistence and Database Queries

We can query the database for history (to list all previous PRDs) and fetch our specific document back by ID to verify it was stored correctly.

In [20]:
print("--- Querying PRD History ---")
history = db_manager.get_history()
print(f"Found {len(history)} documents in history.")
for item in history:
    print(f"ID: {item['id']} | Product: {item['product_idea']} | Generated At: {item['timestamp']}")

print("\n--- Querying Document Details by ID ---")
fetched_doc = db_manager.get_prd_by_id(prd_id)
if fetched_doc:
    print("Successfully retrieved saved document from DB.")
    # Print first few lines of markdown to inspect format
    print("\nMarkdown Preview (First 500 chars):")
    print(fetched_doc['full_prd_markdown'][:500] + "...")

--- Querying PRD History ---
Found 3 documents in history.
ID: 6a3d1cce2fcde7616bc1c1ac | Product: Build a mobile healthcare platform | Generated At: 2026-06-25 12:19:26.573000
ID: 6a3d1ccb2fcde7616bc1c1aa | Product: Build a mobile healthcare platform | Generated At: 2026-06-25 12:19:23.734000
ID: 6a3d1cc32fcde7616bc1c1a8 | Product: Build a mobile healthcare platform | Generated At: 2026-06-25 12:19:15.697000

--- Querying Document Details by ID ---
Successfully retrieved saved document from DB.

Markdown Preview (First 500 chars):
# Product Requirement Document (PRD)

**Product Idea:** Build a mobile healthcare platform
**Target Audience:** Patients seeking remote doctor consultations and prescription tracking
**Key Constraints:** Must be HIPAA compliant, run on iOS and Android
**Generated At:** 2026-06-25 12:19:26 UTC

---

## 1. Problem Statement
No problem statement provided.

## 2. Target Personas
## 3. User Stories
## 4. Functional Requirements
| Module | Feature | Description | 

### Step 7: Clean Up Test Document

We will delete this test document to keep our database clean (optional).

In [21]:
# Uncomment the lines below if you want to test deletion
# success = db_manager.delete_prd(prd_id)
# print(f"PRD deletion success: {success}")